In [ ]:
from pathlib import Path
from markitdown import MarkItDown

# Khởi tạo MarkItDown
md = MarkItDown()

# Chuyển đổi file PDF sáng Markdown
result = md.convert("c4.pptx")

# Lưu thẳng vào thư mục hiện tại
output_file = Path("c4.md")
output_file.write_text(result.text_content, encoding="utf-8")

print(f"Đã lưu thành công: {output_file.resolve()}")

In [ ]:
# -------------------------------------------------------------
# BƯỚC 1: LOAD DỮ LIỆU & TIỀN XỬ LÝ CƠ BẢN
# -------------------------------------------------------------
import pandas as pd
import numpy as np

# Đọc dữ liệu (từ file csv/txt hoặc datasets có sẵn)
df = pd.read_csv('data.csv') 
# Tách đặc trưng X và nhãn y
X = df.drop(columns=['label'])
#y = df['label'] # Hoặc y = df['label'].map({'ham': 0, 'spam': 1}) nếu nhãn là chữ
y = df['label'].map({'h': 0, 'g': 1})
# -------------------------------------------------------------
# BƯỚC 2: CHIA TRAIN / TEST (Luôn nhớ stratify để tránh lệch nhãn)
# -------------------------------------------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# -------------------------------------------------------------
# BƯỚC 3: CHUẨN HÓA / VECTOR HÓA ĐẶC TRƯNG
# -------------------------------------------------------------
# Nếu dữ liệu số (Tabular):
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# HOẶC nếu dữ liệu chữ (Text/Spam):
# from sklearn.feature_extraction.text import TfidfVectorizer
# vec = TfidfVectorizer(stop_words='english', max_features=5000)
# X_train_vec = vec.fit_transform(X_train)
# X_test_vec = vec.transform(X_test)

# -------------------------------------------------------------
# BƯỚC 4: KHỞI TẠO & HUẤN LUYỆN MÔ HÌNH (Fit & Predict)
# -------------------------------------------------------------
from sklearn.tree import DecisionTreeClassifier
# (Hoặc: KNeighborsClassifier, LogisticRegression, MultinomialNB, MLPClassifier, SVC)
model = DecisionTreeClassifier(max_depth=3, random_state=42)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1] # Nếu cần tính ROC-AUC

# -------------------------------------------------------------
# BƯỚC 5: ĐÁNH GIÁ HIỆU NĂNG & TRỰC QUAN HÓA
# -------------------------------------------------------------
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
import pandas as pd 
df = pd.read_excel('datasets/D06 - magic04.data.xlsx','magic04')
df.head()
df.info()
df.describe()

In [ ]:
#X = df.drop(columns = ['11. class'])
#y = df['11. class']
X = df.iloc[:, :10]
y = df.iloc[:, -1].map({'g' : 1, 'h' : 0})
print(X, y)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled =  scaler.transform(X_test)
print(y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state = 42, max_depth = 7)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

import seaborn as sns
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues')

import matplotlib.pyplot as plt
plt.title('CONFUSION MATRIX')
plt.xlabel('Actual')
plt.ylabel('Predict')
plt.show()

In [ ]:
# =============================================================
# BƯỚC 1: LOAD DỮ LIỆU & LẤY MA TRẬN ĐẶC TRƯNG X
# =============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data.csv')
# Bỏ cột nhãn hoặc cột ID (nếu có), chỉ giữ lại các cột số để gom cụm
X = df.drop(columns=['CustomerID', 'label'], errors='ignore') 

# =============================================================
# BƯỚC 2: CHUẨN HÓA DỮ LIỆU (BẮT BUỘC VỚI CLUSTERING)
# =============================================================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =============================================================
# BƯỚC 3: TÌM SỐ CỤM K TỐI ƯU BẰNG PHƯƠNG PHÁP ELBOW (KHỦY TAY)
# =============================================================
from sklearn.cluster import KMeans

inertias = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_) # inertia_ là tổng bình phương khoảng cách (SSE)

# Vẽ đồ thị Elbow
plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='o', color='blue')
plt.title("Phương pháp Elbow tìm số cụm K tối ưu")
plt.xlabel("Số cụm K")
plt.ylabel("Inertia (SSE)")
plt.grid(True)
plt.show()

# =============================================================
# BƯỚC 4: HUẤN LUYỆN K-MEANS VỚI K TỐI ƯU (Ví dụ K = 3)
# =============================================================
optimal_k = 3 # Chọn điểm gập của khuỷu tay
kmeans = KMeans(n_clusters=optimal_k, random_state=42)

# fit_predict vừa học vừa gán nhãn cụm (0, 1, 2,...) cho từng dòng dữ liệu
labels_km = kmeans.fit_predict(X_scaled)

# Lưu nhãn cụm vào lại DataFrame gốc để phân tích
df['Cluster'] = labels_km

# =============================================================
# BƯỚC 5: ĐÁNH GIÁ CHẤT LƯỢNG PHÂN CỤM (SILHOUETTE SCORE)
# =============================================================
from sklearn.metrics import silhouette_score

sil_score = silhouette_score(X_scaled, labels_km)
print(f"Hệ số Silhouette Score: {sil_score:.4f}") 
# (Điểm từ -1 đến 1: Càng gần 1 nghĩa là các cụm phân tách càng rõ ràng, đẹp)

# =============================================================
# BƯỚC 6: TRỰC QUAN HÓA CÁC CỤM (SCATTER PLOT)
# =============================================================
# Vẽ 2 thuộc tính đầu tiên để xem sự phân bố của các cụm
plt.figure(figsize=(8, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_km, cmap='viridis', s=50, alpha=0.8)

# Vẽ các tâm cụm (Centroids) màu đỏ
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, marker='X', label='Tâm cụm (Centroids)')

plt.title("Trực Quan Hóa Kết Quả Phân Cụm K-Means")
plt.xlabel("Đặc trưng 1 (Scaled)")
plt.ylabel("Đặc trưng 2 (Scaled)")
plt.legend()
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

# eps: bán kính láng giềng, min_samples: số điểm tối thiểu để tạo thành cụm
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)

# Điểm có nhãn -1 là điểm NHIỄU (Noise/Outlier)
print("Số cụm tìm được:", len(set(labels_db)) - (1 if -1 in labels_db else 0))
print("Số điểm nhiễu:", list(labels_db).count(-1))